# Notebook 41 — Online Optimization

This notebook demonstrates the online optimization primitives in the Multigen SDK.
All components operate without real LLM calls — rewards are simulated.

| Class | Role |
|---|---|
| `PromptBandit` | Epsilon-greedy multi-armed bandit over prompt variants |
| `FewShotLibrary` | Scored example store with similarity-based retrieval |
| `AgentSpecialisation` | Tracks per-topic EWMA scores to route to expert agents |
| `EpisodicFeedbackLoop` | Mines EpisodicMemory to extract few-shot examples |
| `OptimizationManager` | Unified facade: select → execute → record |

## Setup

In [ ]:
import sys
sys.path.insert(0, '../sdk')

import random
from multigen.optimization import (
    PromptBandit, FewShotLibrary,
    AgentSpecialisation, EpisodicFeedbackLoop,
    OptimizationManager,
)

print('Optimization module loaded successfully')

## Section 1: Prompt Bandit

`PromptBandit` implements an epsilon-greedy multi-armed bandit over prompt variants.
It:
- Explores random variants with probability `epsilon`
- Exploits the highest-reward variant the rest of the time
- Always tries unpulled variants first (cold-start safety)

This lets prompts self-improve from outcome signals without retraining.

In [ ]:
# Define 3 prompt variant templates
templates = [
    'Answer briefly: {question}',
    'Think step by step, then answer: {question}',
    'You are an expert. Provide a detailed answer to: {question}',
]

bandit = PromptBandit(templates, epsilon=0.15)
print(f'Bandit created with {len(bandit.variants())} variants')
for v in bandit.variants():
    print(f'  [{v.id}] {v.template[:60]!r}')

In [ ]:
# Simulate 50 calls with different reward distributions per variant
# Variant 2 (expert) has the highest mean reward
rng = random.Random(42)

reward_map = {
    0: lambda: rng.gauss(0.55, 0.10),  # 'briefly' — OK quality
    1: lambda: rng.gauss(0.72, 0.08),  # 'step by step' — good quality
    2: lambda: rng.gauss(0.88, 0.06),  # 'expert' — best quality
}

for trial in range(50):
    variant = bandit.select()
    # Reward is based on which variant was selected
    variant_index = list({v.id for v in bandit.variants()}).index(variant.id)
    # Map variant_index safely
    all_ids = [v.id for v in bandit.variants()]
    idx = all_ids.index(variant.id)
    reward = max(0.0, min(1.0, reward_map[idx]()))
    bandit.record(variant.id, reward)

print('After 50 simulated trials:')
print('-' * 55)
for stat in bandit.stats():
    print(f'  [{stat["id"]}] pulls={stat["pulls"]:3d}  mean_reward={stat["mean_reward"]:.3f}  win_rate={stat["win_rate"]:.3f}')

In [ ]:
best = bandit.best_variant()
print(f'Best variant after 50 trials:')
print(f'  id         : {best.id}')
print(f'  template   : {best.template!r}')
print(f'  mean_reward: {best.mean_reward:.3f}')
print(f'  pulls      : {best.pulls}')
print(f'  win_rate   : {best.win_rate:.3f}')

# Render the best variant with a real question
rendered = best.render(question='What is the transformer attention mechanism?')
print(f'\nRendered prompt:')
print(f'  {rendered!r}')

In [ ]:
# UCB1 mode — more principled exploration-exploitation tradeoff
bandit_ucb = PromptBandit(templates, epsilon=0.15, ucb=True)

for trial in range(50):
    variant = bandit_ucb.select()
    all_ids = [v.id for v in bandit_ucb.variants()]
    idx = all_ids.index(variant.id)
    reward = max(0.0, min(1.0, reward_map[idx]()))
    bandit_ucb.record(variant.id, reward)

print('UCB1 bandit after 50 trials:')
for stat in bandit_ucb.stats():
    print(f'  [{stat["id"]}] pulls={stat["pulls"]:3d}  mean_reward={stat["mean_reward"]:.3f}')
print(f'Best (UCB): {bandit_ucb.best_variant().template[:50]!r}')

## Section 2: Few-Shot Library

`FewShotLibrary` stores scored input/output examples and retrieves the most
relevant ones for a given query using token-overlap (Jaccard) similarity.

You can replace the `_similarity` method to use vector embeddings for
production-grade semantic retrieval.

In [ ]:
lib = FewShotLibrary(capacity=100, min_score=0.0)

# Add 10 examples with varying quality scores
examples = [
    ('What is machine learning?',         'ML is a subset of AI where systems learn from data.',                   0.95),
    ('Explain neural networks',           'Neural networks are layers of nodes modelling the brain.',              0.92),
    ('What is backpropagation?',           'Backprop computes gradients via the chain rule.',                       0.90),
    ('What is the attention mechanism?',   'Attention weighs the relevance of each token to others.',               0.88),
    ('What is a transformer?',            'A transformer uses multi-head attention and feed-forward layers.',      0.87),
    ('Explain gradient descent',          'Gradient descent minimises loss by moving against the gradient.',       0.85),
    ('What is overfitting?',              'Overfitting means a model memorises training data and fails to generalise.', 0.80),
    ('What is a vector embedding?',       'An embedding maps discrete objects into a continuous vector space.',    0.75),
    ('What is regularisation?',           'Regularisation adds a penalty to reduce model complexity.',             0.70),
    ('What is cross-entropy loss?',       'Cross-entropy measures the difference between predicted and true distributions.', 0.65),
]

for q, a, score in examples:
    lib.add(q, a, score=score, tags=['ml', 'fundamentals'])

print(f'Library size: {lib.size()} examples')

In [ ]:
# Retrieve by query — returns examples ranked by similarity * score
query = 'How does attention work in transformers?'
results = lib.retrieve(query, n=3)

print(f'Query: "{query}"')
print(f'Top {len(results)} retrieved examples:')
print('-' * 60)
for i, ex in enumerate(results, 1):
    print(f'  {i}. score={ex.score:.2f}  input: {ex.input!r}')
    print(f'       output: {ex.output}')

In [ ]:
# Format examples as a few-shot block ready for prompt injection
shots = lib.retrieve('What are embeddings and vector spaces?', n=3)
block = lib.format_shots(shots)
print('Formatted few-shot block:')
print('=' * 60)
print(block)
print('=' * 60)

In [ ]:
# Update score based on downstream feedback (online learning)
lib.update_score('What is overfitting?', delta=+0.10)  # model improved
lib.update_score('What is cross-entropy loss?', delta=-0.05)  # lower quality detected

# Re-retrieve to confirm ordering reflects updated scores
results2 = lib.retrieve('machine learning concepts', n=5)
print('Updated rankings (by similarity * score):')
for i, ex in enumerate(results2, 1):
    print(f'  {i}. score={ex.score:.2f}  {ex.input}')

## Section 3: Agent Specialisation

`AgentSpecialisation` tracks per-topic performance for each agent using an
Exponentially Weighted Moving Average (EWMA). It then routes queries to the
agent with the highest score for each topic.

This is how a multi-agent system can self-organise into specialists over time.

In [ ]:
spec = AgentSpecialisation(
    agent_names=['coder_agent', 'analyst_agent', 'writer_agent'],
    alpha=0.2,  # EWMA smoothing factor
)

# Simulate performance records for 3 agents across 3 topics
# coder_agent is best at coding, analyst_agent at data analysis, writer_agent at writing
records = [
    # (agent, topic, score)
    ('coder_agent',   'coding',        0.95),
    ('coder_agent',   'coding',        0.92),
    ('coder_agent',   'data_analysis', 0.60),
    ('coder_agent',   'writing',       0.55),

    ('analyst_agent', 'data_analysis', 0.91),
    ('analyst_agent', 'data_analysis', 0.88),
    ('analyst_agent', 'coding',        0.70),
    ('analyst_agent', 'writing',       0.65),

    ('writer_agent',  'writing',       0.93),
    ('writer_agent',  'writing',       0.90),
    ('writer_agent',  'data_analysis', 0.62),
    ('writer_agent',  'coding',        0.58),
]

for agent, topic, score in records:
    spec.record(agent, topic, score)

print('Performance records ingested')

In [ ]:
# Show rankings per topic
topics = ['coding', 'data_analysis', 'writing']

for topic in topics:
    best = spec.best_agent_for(topic)
    rankings = spec.rankings(topic)
    print(f'\nTopic: {topic!r}  ->  Best: {best}')
    for rank_i, (agent, score) in enumerate(rankings, 1):
        marker = ' <<' if agent == best else ''
        print(f'  {rank_i}. {agent:<20} score={score:.3f}{marker}')

In [ ]:
# Full performance profile for each agent
for agent_name in ['coder_agent', 'analyst_agent', 'writer_agent']:
    profile = spec.profile(agent_name)
    print(f'\nProfile: {profile["agent"]}')
    for topic, info in profile['topics'].items():
        print(f'  {topic:<20} score={info["score"]:.3f}  count={info["count"]}')

## Section 4: Episodic Feedback Loop

`EpisodicFeedbackLoop` mines an `EpisodicMemory` store to extract high-quality
episodes as few-shot examples. Episodes with `score >= min_score` are added to
the `FewShotLibrary` automatically.

This creates a self-improving loop: good outcomes become training examples.

In [ ]:
# Mock EpisodicMemory that stores episode objects with a .content dict
class MockEpisode:
    def __init__(self, input_text, output_text, score):
        self.content = {
            'input': input_text,
            'output': output_text,
            'score': score,
        }

class MockEpisodicMemory:
    def __init__(self):
        self._episodes = []

    def add(self, episode):
        self._episodes.append(episode)

    def recent(self, n=100):
        return self._episodes[-n:]

    def __len__(self):
        return len(self._episodes)

# Simulate recorded episodes from a running agent
episodic_memory = MockEpisodicMemory()

episode_data = [
    ('What is a language model?',      'A language model assigns probability distributions over sequences of tokens.', 0.92),
    ('Explain tokenisation',           'Tokenisation splits text into subword units called tokens.',                   0.88),
    ('What is fine-tuning?',           'Fine-tuning adapts a pre-trained model to a specific task with labelled data.', 0.85),
    ('What is zero-shot learning?',    'Zero-shot learning uses a model without task-specific training examples.',       0.80),
    ('What is a prompt?',              'A prompt is the input text given to a language model to guide its output.',     0.78),
    ('What is hallucination in LLMs?', 'Hallucination is when an LLM generates plausible but factually incorrect text.', 0.75),
    ('Explain chain-of-thought',       'Chain-of-thought prompting asks the model to reason step by step.',             0.72),
    ('What is in-context learning?',   'In-context learning lets LLMs adapt to tasks using examples in the prompt.',    0.68),
    ('What is a temperature parameter?', 'Temperature controls the randomness of token sampling.',                     0.60),
    ('What is a base model?',          'Decent but short answer.',                                                      0.40),  # low quality — should be filtered
]

for q, a, score in episode_data:
    episodic_memory.add(MockEpisode(q, a, score))

print(f'EpisodicMemory populated with {len(episodic_memory)} episodes')

In [ ]:
# Create a library and feedback loop
feedback_library = FewShotLibrary(capacity=200)
feedback_bandit  = PromptBandit(['Answer: {question}', 'Explain in detail: {question}'])

loop = EpisodicFeedbackLoop(
    episodic_memory=episodic_memory,
    library=feedback_library,
    bandit=feedback_bandit,
    min_score=0.65,   # only mine episodes with score >= 0.65
    mine_every_n=5,
)

print(f'EpisodicFeedbackLoop created')
print(f'  min_score threshold : {loop.min_score}')
print(f'  mine_every_n        : {loop.mine_every_n}')

In [ ]:
# Mine all episodes from memory
added = loop.mine(n=100)
stats = loop.stats()

print(f'Mining complete!')
print(f'  Episodes in memory   : {len(episodic_memory)}')
print(f'  Examples added       : {added}')
print(f'  Library size         : {stats["library_size"]}')
print(f'  examples_mined total : {stats["examples_mined"]}')
print()
print(f'(Episodes with score < {loop.min_score} were filtered out)')

In [ ]:
# The library is now populated — retrieve examples for a new query
mined_examples = feedback_library.retrieve('How do language models learn?', n=3)
print('Retrieved examples from mined library:')
for i, ex in enumerate(mined_examples, 1):
    print(f'  {i}. [score={ex.score:.2f}] Input: {ex.input}')
    print(f'       Output: {ex.output[:80]}...' if len(ex.output) > 80 else f'       Output: {ex.output}')

## Section 5: OptimizationManager — Unified Workflow

`OptimizationManager` combines all four components into a single facade:
1. `select_prompt()` — picks the best prompt variant
2. `get_few_shots()` — retrieves relevant examples
3. `best_agent()` — routes to the best specialist
4. `record_outcome()` — updates all components from a single call

This is the recommended entry point for production use.

In [ ]:
mgr = OptimizationManager(
    prompt_templates=[
        'Answer concisely: {question}',
        'Think step by step about: {question}',
        'You are an expert. Answer: {question}',
    ],
    agent_names=['fast_agent', 'balanced_agent', 'quality_agent'],
    library_capacity=200,
    bandit_epsilon=0.15,
)

print('OptimizationManager created')
print(f'  Prompt variants : {len(mgr.bandit.variants())}')
print(f'  Agent names     : {[a for a in ["fast_agent", "balanced_agent", "quality_agent"]]}')

In [ ]:
# Simulate 20 rounds: select → (mock) execute → record
# quality_agent consistently gets higher scores on 'ml' topic
rng2 = random.Random(99)

agent_score_map = {
    'fast_agent':     lambda topic: rng2.gauss(0.60, 0.08),
    'balanced_agent': lambda topic: rng2.gauss(0.75, 0.07),
    'quality_agent':  lambda topic: rng2.gauss(0.89, 0.05),
}

questions = [
    'What is a neural network?', 'Explain backpropagation', 'What is overfitting?',
    'Describe attention mechanism', 'What is a transformer?', 'Explain fine-tuning',
    'What is zero-shot learning?', 'Describe tokenisation', 'What is RAG?',
    'Explain chain-of-thought prompting',
]

print('Round | Variant ID | Agent          | Score | Best so far')
print('-' * 70)

for round_i in range(20):
    q = questions[round_i % len(questions)]

    # Step 1: select prompt variant
    variant = mgr.select_prompt()

    # Step 2: determine best agent for this topic
    agent_name = mgr.best_agent('ml')

    # Step 3: mock execution
    score = max(0.0, min(1.0, agent_score_map[agent_name]('ml')))
    mock_output = f'Mock answer to: {q[:30]}...'

    # Step 4: record outcome (updates bandit + specialisation + library)
    mgr.record_outcome(
        score=score,
        variant_id=variant.id,
        agent=agent_name,
        topic='ml',
        input_text=q,
        output_text=mock_output,
    )

    best_v = mgr.bandit.best_variant()
    print(f'  {round_i+1:2d}  | {variant.id}   | {agent_name:<15} | {score:.3f} | {best_v.id} ({best_v.mean_reward:.3f})')

In [ ]:
# Final stats across all components
stats = mgr.stats()

print('Final OptimizationManager stats:')
print('\nPrompt Bandit rankings:')
for s in stats['bandit']:
    print(f'  [{s["id"]}] pulls={s["pulls"]:2d}  mean_reward={s["mean_reward"]:.3f}  win_rate={s["win_rate"]:.3f}')

print(f'\nFew-shot library size: {stats["library_size"]}')

print('\nAgent specialisation for topic "ml":')
for agent_name, score in mgr.specialisation.rankings('ml'):
    print(f'  {agent_name:<20} score={score:.3f}')

best_prompt = mgr.bandit.best_variant()
best_agent_ml = mgr.best_agent('ml')
print(f'\nBest prompt : {best_prompt.template!r}')
print(f'Best agent  : {best_agent_ml}')

## Summary

```python
from multigen.optimization import OptimizationManager

# One-time setup
mgr = OptimizationManager(
    prompt_templates=['Be concise: {q}', 'Think step by step: {q}'],
    agent_names=['fast_agent', 'quality_agent'],
)

# Inference loop
for query in incoming_queries:
    variant = mgr.select_prompt()
    shots   = mgr.get_few_shots(query, n=3)
    agent   = mgr.best_agent(topic)

    # Build prompt
    prompt = variant.render(q=query)
    if shots:
        prompt = mgr.library.format_shots(shots) + '\n\n' + prompt

    # Run agent (mocked here)
    response, score = run_agent(agent, prompt)

    # Record outcome — updates everything
    mgr.record_outcome(
        score=score,
        variant_id=variant.id,
        agent=agent,
        topic=topic,
        input_text=query,
        output_text=response,
    )
```

**Key properties:**
- `PromptBandit` converges on best template within ~50 trials
- `FewShotLibrary` uses Jaccard similarity; swap `_similarity` for embeddings
- `AgentSpecialisation` uses EWMA (`alpha=0.1–0.2`) for smooth score tracking
- `OptimizationManager.record_outcome()` updates all three components in one call